In [1]:
!pip install --quiet pymupdf faiss-cpu sentence-transformers transformers
!pip install rapidfuzz
!pip install PyMuPDF


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 24.7 MB/s eta 0:00:00


In [2]:
import os
import re
import json
import pickle
from math import ceil

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from collections import deque



In [6]:
# Step 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
layer1_dir = "/content/drive/MyDrive/Rag Wbg/Fixed/layer1"
layer2_dir = "/content/drive/MyDrive/Rag Wbg/Fixed/layer2"

In [8]:
def query_layer(question, index, model, chunks, top_k=5, multiplier=5, merge_keywords=False):
    """
    Generic query function for Layer 1 and Layer 2.
    If merge_keywords=True, merges keywords per (part, section, page).
    Returns both final results and a formatted raw_context string.
    """
    # Encode and search
    q_emb = model.encode([question], convert_to_numpy=True)
    D, I = index.search(q_emb, top_k * multiplier)
    distances = D[0]

    # Collect raw results
    raw_results = []
    for idx, dist in zip(I[0], distances):
        item = chunks[idx]
        entry = {
            "part": item["part"],
            "section": item["section"],
            "page": item["page"],
            "context": item["context"],
            "distance": dist
        }
        if merge_keywords:
            entry["keyword"] = item["keyword"]
        raw_results.append(entry)

    # Normalize confidence
    min_d, max_d = min(distances), max(distances)
    for r in raw_results:
        r["confidence"] = 1 - (r["distance"] - min_d) / (max_d - min_d + 1e-6)

    # Deduplicate
    merged = {}
    for r in raw_results:
        key = (r["part"], r["section"], r["page"])
        if key not in merged:
            merged[key] = {"part": r["part"], "section": r["section"], "page": r["page"],
                           "context": r["context"], "confidence": r["confidence"]}
            if merge_keywords:
                merged[key]["keywords"] = {r["keyword"]}
        else:
            if r["confidence"] > merged[key]["confidence"]:
                merged[key]["context"] = r["context"]
                merged[key]["confidence"] = r["confidence"]
            if merge_keywords:
                merged[key]["keywords"].add(r["keyword"])

    # Prepare final results
    final_results = []
    for v in merged.values():
        res = {"part": v["part"], "section": v["section"], "page": v["page"],
               "context": v["context"], "confidence": v["confidence"]}
        if merge_keywords:
            res["keyword"] = ", ".join(sorted(v["keywords"]))
        final_results.append(res)

    final_results.sort(key=lambda x: x["confidence"], reverse=True)
    final_results = final_results[:top_k]

    # Build raw_context
    raw_context = "".join(
        f"[{r['part']}] Section: {r['section']} | Page: {r['page']}"
        + (f" | KW: {r['keyword']}" if merge_keywords else "")
        + f" | Conf: {r['confidence']:.3f}\n{r['context']}\n----\n"
        for r in final_results
    )

    return final_results, raw_context



# Load artifacts
def load_layer_artifacts(base_dir, index_file, chunks_file):
    index = faiss.read_index(os.path.join(base_dir, index_file))
    with open(os.path.join(base_dir, chunks_file), "rb") as f:
        chunks = pickle.load(f)
    return index, chunks

index1, chunks1 = load_layer_artifacts(layer1_dir, "layer1_faiss_index.bin", "layer1_chunks.pkl")
index2, chunks2 = load_layer_artifacts(layer2_dir, "layer2_faiss_index.bin", "layer2_chunks.pkl")


# Store last 3 responses
history = deque(maxlen=3)

def clean_references(my_text: str) -> str:
    """
    Removes the first 'References:' section from the text,
    keeping the Q/A part and the last 'References:' section.
    """
    parts = my_text.split("References:")
    if len(parts) >= 3:
        # keep Q/A (before first refs) + last references section
        return parts[0].strip() + "\n\nReferences:" + parts[-1].strip()
    return my_text.strip()  # fallback if only one References section


def add_point_numbering_safeguard(answer_text, raw_context):
    """
    Ensures each snippet starts with a point numbering (e.g., 2.6.13.10.1.).
    If missing, tries to extract from snippet itself or raw_context.
    """
    pattern = re.compile(r"\b\d{1,2}(?:\.\d{1,2}){3,5}\b")

    fixed_lines = []
    for line in answer_text.splitlines():
        if line.strip().startswith("- Snippet:"):
            snippet_text = line.split("Snippet:", 1)[1].strip().strip('"')

            # Check if snippet already starts with a numbering
            if not re.match(pattern, snippet_text):
                match = pattern.search(snippet_text)
                if match:
                    # Found inside snippet
                    numbering = match.group(0)
                    snippet_text = f"{numbering} {snippet_text}"
                else:
                    # Fallback: search in raw_context near snippet text
                    idx = raw_context.find(snippet_text[:30])  # first 30 chars
                    if idx != -1:
                        context_before = raw_context[max(0, idx-100):idx]
                        match2 = pattern.findall(context_before)
                        if match2:
                            numbering = match2[-1]  # nearest preceding number
                            snippet_text = f"{numbering} {snippet_text}"

            fixed_lines.append(f'- Snippet: "{snippet_text}"')
        else:
            fixed_lines.append(line)

    return "\n".join(fixed_lines)


def generate_answer_with_references(doc_name, raw_context, query):
    """
    Generates a structured answer with references in the desired format,
    with safeguard to enforce numbering in snippets.
    Always uses the last 3 answers as rolling history for continuity.
    """
    # Build history context
    related_context = ""
    if history:
        related_context = "\n\n".join(
            [f"Previous Q: {q}\nPrevious A: {a}" for q, a in history]
        )

    prompt = f"""
You are a structural engineer with 30 years of experience. Use the following document text to answer the question.
Provide a clear answer and include references from the text.

Important instructions:
1. In the "Answer", summarize the information from ALL relevant parts of the document.
2. In the "References" section, list EVERY distinct segment of the text that contributed.
3. In every "Snippet", always begin with the exact point/numbering (e.g., "2.6.13.10.1.") as it appears in the source text.
4. Do not skip or compress references.
5. Use the last answers (if provided) as context for continuity.

EXAMPLES OF GOOD RESPONSES:

Example 1 - Question with Multiple Sources:
Question: Are wind load calculations specified for different building components?

Answer: Yes, the Rwanda Building Code specifies wind load calculations for different building components. For general structures, all wind actions acting on buildings must be calculated in accordance with RS 114-2, with grades of exposure to wind following Table 3 in RS 114-2 [2.6.2.3.1.2., Page 194]. For specific components like cladding and glazing, additional requirements apply where cladding structural components must be designed to resist wind forces from close places, and glazing components for windows, doors, and curtain walls must be designed to resist wind loads [2.6.13.10.1., Page 271]. Engineers designing glazing structures must consider wind speed calculations along with earthquake zone factors and correlation factors [2.6.13.11., Page 271].

References:
- Document: Rwanda Building Code, Part: 6, Section: LOADS, FORCES & EFFECTS, Page: 194
- Snippet: "2.6.2.3.1.2. The grades of exposure to wind shall be in accordance with Table 3 in RS114-2. When designing buildings, structures and any other structural components, the wind load shall be considered. All the wind actions acting on the building shall be calculated in accordance with RS 114-2."

- Document: Rwanda Building Code, Part: 6, Section: CLADDING & GLAZING, Page: 271
- Snippet: "2.6.13.10.1. Cladding structural components that are either directly or indirectly loaded by wind forces from the close places shall be in the way to resist those forces system. 2.6.13.11. The engineer in charge of designing glazing structures shall consider the wind speed calculations and the earthquake prone zone as well as all factors of correlations."

Example 2 - Complex Multi-Part Question:
Question: What are the fire resistance requirements for structural elements?

Answer: The Rwanda Building Code establishes comprehensive fire resistance requirements for structural elements based on building occupancy and height. Primary structural frame elements must have fire resistance ratings ranging from 1 to 3 hours depending on the building type [3.7.2.1., Page 156]. Load-bearing walls require fire resistance ratings between 1 to 4 hours based on occupancy classification [3.7.2.2., Page 157]. Floor and roof assemblies must meet specific fire resistance criteria, with ratings typically ranging from 1 to 2 hours [3.7.2.3., Page 158]. All fire resistance ratings must be determined through standard fire tests conducted in accordance with ASTM E119 or equivalent standards [3.7.1.5., Page 155].

References:
- Document: Rwanda Building Code, Part: 3, Section: FIRE PROTECTION, Page: 156
- Snippet: "3.7.2.1. Primary structural frame elements including columns, beams, and trusses shall have fire resistance ratings as specified in Table 7.2 based on occupancy type and building height."

Use the following output format:

Question: <Your Question>
Answer: <Answer text with reference markers [Source: Section, Page]>

References:
For each reference used, provide:
- Document: {doc_name}, Part: <Part>, Section: <Section>, Page: <Page>
- Snippet: "point number. snippet text ..."

{related_context}

Document text for reference:
\"\"\"{raw_context}\"\"\"

Question: {query}
"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    raw_answer = response.choices[0].message.content

    # Apply safeguard to ensure numbering
    final_answer = add_point_numbering_safeguard(raw_answer, raw_context)

    # Save Q/A to history (max 3)
    history.append((query, final_answer))

    #final_answer = clean_references(final_answer)

    return final_answer





In [9]:
# Example usage
doc_name = "Rawanda Building Code"

In [ ]:
query = "Is the scope and application of the code specified? If yes, does it cover the following? i. Buildings (defined as structures intended for the use or occupancy of people or for shelter – both temporary and permanent, including structures that may be moved or relocated)."

In [ ]:
query = "If yes, does it cover the following? ii. Some aspects covering alterations to existing buildings? (Defined as renovations, modifications, change of use, additions, strengthening and/or rehabilitation – see also questions in section B6)."

In [ ]:
query = "Are importance classifications for buildings defined?"

In [ ]:
query = "Are minimum dead load assumptions defined?"

In [ ]:
query = "Are live loads provided for all anticipated occupancies (for floors, basements and roofs): i. Uniformly distributed loads;"

In [ ]:
query = "Is there country-specific wind speed data for design?"

In [ ]:
query = "If yes, please specify how the design information is provided: • Wind speed map? • Wind speed by zonation map? • Wind speed by city or region name? • Other (please specify)?"

In [ ]:
query = "Are there Importance Factors for wind design? If yes, are they based upon (note all that apply): i. Building occupancy? ii. Building size? iii. Building risk level? iv. Building usage type? v. Other (please specify)?"


In [ ]:
query = "Do the provisions provide a procedure to determine design wind pressures for the main load resisting system?"

In [ ]:
query = "Does the code and/or standards have any requirements related to specifying seismic design loads/design seismic hazard criteria?"

In [ ]:
query = "Is there a definition of the type and amount of dead and live loads to include in the seismic mass/weight of the building for the modelling and calculation of the lateral design loading?"


In [ ]:
query = "Are country-specific seismic hazard parameters provided for design? i. If yes, when were these last updated?"

In [ ]:
query = "Are country-specific seismic hazard parameters provided for design? ii. If yes, how are seismic hazard parameters specified(Please note all that apply) • in the form of seismic hazard maps • by PGA value or spectral acceleration values for city or other geographic subset of the country • by PGA value or spectral acceleration values for the entire country • other – please specify in comments"

In [ ]:
query = "Are country-specific seismic hazard parameters provided for design? i. If yes, when were these last updated?"

In [ ]:
query = "Is the effect of soil site conditions considered in the seismic design, for example, by specifying soil modification factors linked to soil type and adjusting the design response spectra by these factors?"

In [ ]:
query = "Are there requirements for geotechnical site investigations depending on the site location, scale and type of construction?"

In [ ]:
query = "Are there provisions for the calculation of geotechnical design parameters based on site investigation information?"

In [ ]:
query = "Are there design and detailing requirements for shallow foundations, including ground deformation?"

In [ ]:
query = "Are there design and detailing requirements for deep foundations including ground deformations?"

In [ ]:
query = "Are design procedures prescribed for different types of retaining structures, including reinforced earth? Please specify."

In [ ]:
query = "What types of construction materials are addressed by the code, including concrete (B4.3), structural steel (B4.4), masonry (B4.5), timber (B4.6), earthen construction (B4.7), bamboo, vegetative construction, wattle and daub, aluminum, and glass?"

In [ ]:
query = "Do the seismic design provisions address the following topics: i. plan regularity/irregularity, ii. vertical regularity/irregularity, iii. torsional regularity/irregularity, iv. redundancy requirements for the lateral load-resisting system?"

In [ ]:
query = "Are there provisions related to the seismic design of diaphragms to ensure that load can be transferred to vertical elements of the lateral load-resisting system?"

In [ ]:
query = "Do the provisions include modification factors for forces and displacements depending on the expected behavior of the type of lateral load-resisting system?"

In [ ]:
query = "Do the criteria include Importance Factors for seismic design?"

In [ ]:
query = "Are there provisions related to drift limits under seismic loading?"

In [ ]:
query = "Are there provisions for the seismic design of non- structural components?"

In [ ]:
query = "Are there requirements for the design of the following types of concrete structures (if by reference, please state standard[s]): i. Reinforced concrete (rc), ii. precast concrete (pcc), iii. Post-tensioned (PT)?"

In [ ]:
query = "Do design provisions include seismic design and detailing of concrete structures?"

In [ ]:
query = "Are there requirements for engineering analysis and design of steel structures?"

In [ ]:
query = "Do design provisions include seismic design and detailing?"

In [ ]:
query = "Are there design requirements for unreinforced masonry?"

In [ ]:
query = "Do requirements cover seismic design and detailing?"

In [ ]:
query = "Are there requirements for engineering analysis and design of timber structures?"

In [ ]:
query = "Are there any provisions for seismic design and detailing of timber structures?"

In [ ]:
query = "Does the building code cover the design of: i. alterations, ii. additions, iii. building renovations/rehabilitation, iv. change of use and/or occupancy, v. seismic retrofit, vi. other types of retrofit including structural improvements (see section B1.1, scope)?"

In [ ]:
query = "Does the building code cover assessment of building vulnerability and/or damage?"

In [ ]:
query = "Do the assessment procedures include provisions related to seismic assessment?"

In [ ]:
query = "Are there provisions for the selection and design of rehabilitation and retrofitting measures?"

In [ ]:
query = " i. If yes, are seismic retrofitting provisions and procedures included?"

In [ ]:
query = "Does the code specify what type or size of addition requires building code compliance?"

**Miscellaneous testing**

In [10]:
query = "are important structural desgin considerations defined?"

In [18]:
query = "Is there country-specific wind speed data for design?"

In [16]:
query = " If yes, please specify how the design information is provided: • Wind speed map? • Wind speed by zonation map? • Wind speed by city or region name? • Other (please specify)?"

In [19]:
query = "Is there a definition of the type and amount of dead and live loads to include in the seismic mass/weight of the building for the modelling and calculation of the lateral design loading?"


In [21]:
query = "Is there a definition of the type and amount of dead and live loads to include in the seismic mass/weight of the building for the modelling and calculation of the lateral design loading? Give me answer that are only relevant to seismic design"


In [23]:
query = "Is there a definition of the type and amount of dead and live loads to include in the seismic mass/weight of the building for the modelling and calculation of the lateral design loading? Look specifically for methodologies that define how to calculate seismic mass (e.g., percentages of live loads to include, treatment of different occupancy types). If not explicitly defined in the main code text, indicate whether this should be found in referenced seismic design standards."

In [24]:
# Load model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Query both layers
results_1, raw_context_1 = query_layer(query, index1, model, chunks1, top_k=5, merge_keywords=True)
results_2, raw_context_2 = query_layer(query, index2, model, chunks2, top_k=5, merge_keywords=False)

# Combine raw_contexts
raw_context = raw_context_1 + raw_context_2


final_answer = generate_answer_with_references(doc_name, raw_context, query)
print(final_answer)


Answer:  
Yes, the Rwanda Building Code defines the types and amounts of dead and live loads to be included in the seismic mass or weight of the building for modeling and calculation of lateral design loading, though detailed methodologies for load percentages by occupancy type are not explicitly provided in the main code text and are expected to be found in the referenced seismic design standards.

Specifically, dead loads must be calculated based on the weights of all building materials according to RS 106 and RS 114-1, including permanent partitions and tanks when full [2.6.2.1.1. to 2.6.2.1.4., Page 191]. For live (imposed) loads, the code requires that imposed loads on floors, stairs, and other structural elements be derived according to RS 106 and RS 114-1, adopting the more onerous values [2.6.2.2.1., Page 191]. Floors must be designed to carry both distributed and concentrated imposed loads [2.6.2.2.2., Page 191].

Regarding seismic mass specifically, the code states that build